# Supplementary Figure 1

Plot example task trials for each task.

Figure 1.c and Supplementary Figure 1

In [1]:
import numpy as np
import math
from pathlib import Path
from jax import numpy as jnp
import jax
from functools import partial

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpecFromSubplotSpec
from matplotlib.patches import Patch

plt.rcParams["svg.fonttype"] = "none"

from nntp.datasets import DataSetManager

from common import (
    subtitle_fontsize,
    panel_indexing_fontsize,
    feature_size,
    output_size,
    CHANNEL,
    CHANNEL_NAME_MAPPING,
    COLORS,
)

In [2]:
COLORS = {
    "fixation": "#7A52CC",  # purple
    "stimulus_1": "#2CA02C",  # green
    "stimulus_2": "#1F77B4",  # blue
    "target": "#D62728",  # red
    "cue": "#4D4D4D",  # dark gray
}

legend_handles = [
    Line2D([0], [0], color=COLORS["fixation"], label="Fixation input"),
    Line2D([0], [0], color=COLORS["stimulus_1"], label="Stimulus 1"),
    Line2D([0], [0], color=COLORS["stimulus_2"], label="Stimulus 2"),
    Line2D([0], [0], color=COLORS["target"], label="Target"),
    Line2D([0], [0], color=COLORS["cue"], label="Cue"),
]

styles = {
    2: ("orange", "Stimulus", "darkorange"),
    3: ("lightgreen", "Delay", "tab:green"),
    4: ("lightcoral", "Response", "tab:red"),
}


color_legend_handles = [
    Patch(
        facecolor=facecolor,
        edgecolor="none",
        label=label,
    )
    for facecolor, label, edgecolor in styles.values()
]

In [3]:
@partial(jax.jit, static_argnames=("stimulus_channel_start", "stimulus_channel_end"))
def compute_phase_mask(X, M, stimulus_channel_start, stimulus_channel_end, amplitude=2):
    """
    Region partition:
        0 = mask
        1 = context
        2 = stimulus
        3 = memory
        4 = response
    """
    mask = M[:, :, 0]  # (T, B)
    T, B = mask.shape
    new_mask = jnp.zeros((T, B))

    # -------------------------------------------------
    # 1. RESPONSE REGION
    # -------------------------------------------------
    masked_for_max = jnp.where(mask != 0, mask, -jnp.inf)  # (T, B)
    resp_vals = jnp.max(masked_for_max, axis=0)  # (B,)
    region_resp = mask == resp_vals[None, :]  # (T, B)

    # -------------------------------------------------
    # 2. INPUT REGION (context + stim + memory) (T, B)
    # -------------------------------------------------
    region_input = (mask != 0) & (~region_resp)

    # -------------------------------------------------
    # 3. STIMULUS REGION (T, B)
    # -------------------------------------------------
    stimulus_vals = jnp.sum(
        jnp.abs(X[:, :, stimulus_channel_start : (stimulus_channel_end + 1)]), axis=2
    )
    region_stim = (stimulus_vals >= amplitude) & region_input

    # -------------------------------------------------
    # 4. DETECT WHETHER STIM EXISTS (B,)
    # -------------------------------------------------
    stim_start = jnp.argmax(region_stim, axis=0)
    stim_exists = jnp.any(region_stim, axis=0)

    # -------------------------------------------------
    # 5. MEMORY
    # -------------------------------------------------
    time = jnp.arange(T)[:, None]
    region_context = (time < stim_start) & region_input
    region_memory = (time >= stim_start) & region_input
    region_memory = region_memory & stim_exists[None, :]

    # -------------------------------------------------
    # 6. COMBINE (priority order)
    # -------------------------------------------------
    new_mask = jnp.where(region_input, 1, new_mask)
    new_mask = jnp.where(region_context, 1, new_mask)
    new_mask = jnp.where(region_memory, 3, new_mask)
    new_mask = jnp.where(region_stim, 2, new_mask)
    new_mask = jnp.where(region_resp, 4, new_mask)
    return new_mask  # (T, B)

In [4]:
def add_phase_shading(ax, mask, text=None):
    """Shade continuous task phases. Phase 0 is ignored."""

    mask = np.asarray(mask)
    if mask.size == 0:
        return

    # Boundaries of continuous regions: [start, end)
    boundaries = np.r_[0, np.flatnonzero(mask[1:] != mask[:-1]) + 1, len(mask)]

    for start, end in zip(boundaries[:-1], boundaries[1:]):
        phase = int(mask[start])

        if phase not in styles:
            continue

        color, label, text_color = styles[phase]
        ax.axvspan(start - 0.5, end - 0.5, color=color, alpha=0.4, zorder=0)

        if text:
            ax.text(
                (start + end - 1) / 2,
                1,
                label,
                ha="center",
                va="bottom",
                transform=ax.get_xaxis_transform(),
                alpha=0.8,
                color=text_color,
                fontsize=10,
                clip_on=False,
            )

In [5]:
def plot_trial_axes(
    parent_ax,
    trial_index,
    X,
    Y,
    M,
    stimulus_1_start,
    stimulus_2_start,
    stimulus_2_end,
    title,
    show_ylabels=True,
    show_xlabel=True,
    text=None,
):
    """
    parent_ax: 外层大图里的一个 placeholder axis。
               函数会删除它，并在它的位置创建 3 个纵向子 axes。
    return: inner_axes, shape=(3,)
    """

    fig = parent_ax.figure
    subplotspec = parent_ax.get_subplotspec()

    # 删除外层 placeholder axis
    parent_ax.remove()

    # 在这个 cell 里面创建 3 个子图
    inner_gs = GridSpecFromSubplotSpec(
        3, 1, subplot_spec=subplotspec, height_ratios=[1, 2, 1], hspace=0.05
    )

    axes = np.empty(3, dtype=object)
    axes[0] = fig.add_subplot(inner_gs[0, 0])
    axes[1] = fig.add_subplot(inner_gs[1, 0], sharex=axes[0])
    axes[2] = fig.add_subplot(inner_gs[2, 0], sharex=axes[0])

    # ---- slice one trial ----
    trial_X = X[:, trial_index, :]
    trial_Y = Y[:, trial_index, :]
    trial_mask = compute_phase_mask(X, M, stimulus_1_start, stimulus_2_end)[
        :, trial_index
    ]

    # 找到最后一个非 padding 时间点
    valid_indices = np.flatnonzero(np.asarray(trial_mask) != 0)

    if valid_indices.size > 0:
        trial_start = valid_indices[0]
        trial_end = valid_indices[-1] + 1
    else:
        trial_start = 0
        trial_end = trial_mask.shape[0]

    # 同时删除开头和结尾的 mask/padding
    trial_X = trial_X[trial_start:trial_end]
    trial_Y = trial_Y[trial_start:trial_end]
    trial_mask = trial_mask[trial_start:trial_end]

    # 横轴从 0 重新开始
    time = np.arange(trial_end - trial_start)

    # ---- break down ----
    fixation_X = trial_X[:, 0]
    stimulation_ring_X_1 = trial_X[:, stimulus_1_start:stimulus_2_start]
    stimulation_ring_X_2 = trial_X[:, stimulus_2_start : (stimulus_2_end + 1)]
    cues_X = trial_X[:, (stimulus_2_end + 1) :]

    fixation_Y = trial_Y[:, 0]
    stimulation_ring_Y = trial_Y[:, 1:]

    # ================= title / fixation =================
    axes[0].set_title(
        CHANNEL_NAME_MAPPING[title],
        fontsize=subtitle_fontsize,
        fontweight="bold",
        loc="left",
        pad=16,
        x=-0.05,
    )

    sns.lineplot(
        x=time, y=fixation_X, ax=axes[0], color=COLORS["fixation"], linewidth=1
    )
    sns.lineplot(x=time, y=fixation_Y, ax=axes[0], color=COLORS["target"], linewidth=1)
    add_phase_shading(axes[0], trial_mask, text)
    axes[0].set_yticks([0, 1])

    # ================= stimulus ring =================
    axes[1].plot(time, stimulation_ring_X_1, color=COLORS["stimulus_1"], linewidth=1)
    axes[1].plot(time, stimulation_ring_X_2, color=COLORS["stimulus_2"], linewidth=1)
    axes[1].plot(time, stimulation_ring_Y, color=COLORS["target"], linewidth=1)
    add_phase_shading(axes[1], trial_mask)

    # ================= cues =================
    axes[2].plot(time, cues_X, color=COLORS["cue"], linewidth=1)
    add_phase_shading(axes[2], trial_mask)
    axes[2].set_yticks([0, 1])

    # ================= labels =================
    if show_ylabels:
        axes[0].set_ylabel("Fixation")
        axes[1].set_ylabel("Stimulus")
        axes[2].set_ylabel("Cues")
        fig.align_ylabels(axes)

    if show_xlabel:
        axes[2].set_xlabel("Time step")

    # ================= styles =================
    axes[0].tick_params(axis="x", which="both", bottom=False, labelbottom=False)
    axes[1].tick_params(axis="x", which="both", bottom=False, labelbottom=False)

    # x 轴只显示开始和结束
    if time.size > 0:
        axes[2].set_xticks([time[0], time[-1]])
        axes[2].set_xticklabels([str(time[0]), str(time[-1])])

    for i, ax in enumerate(axes):
        sns.despine(
            ax=ax,
            top=True,
            right=True,
            left=False,
            bottom=(i < 2),
            trim=True,
            offset=5,
        )

    return axes

In [6]:
def save_trial_axes_png(
    X,
    Y,
    M,
    out_path,
    title,
    trial_index=0,
    stimulus_1_start=1,
    stimulus_2_start=32,
    stimulus_2_end=64,
    figsize=(8, 3),
):
    out_path = Path(out_path)
    out_path.mkdir(parents=True, exist_ok=True)

    fig, parent_ax = plt.subplots(1, 1, figsize=figsize)

    inner_axes = plot_trial_axes(
        parent_ax=parent_ax,
        trial_index=trial_index,
        X=X,
        Y=Y,
        M=M,
        stimulus_1_start=stimulus_1_start,
        stimulus_2_start=stimulus_2_start,
        stimulus_2_end=stimulus_2_end,
        title=title,
        text=True,
    )

    fig.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.55, 0.98),
        ncol=5,
        frameon=False,
        handlelength=2.0,
        columnspacing=1.8,
        borderaxespad=0,
    )

    fig.tight_layout()
    # fig.savefig(out_path / f"{title}.png", dpi=300, bbox_inches="tight")
    fig.savefig(out_path / f"{title}.svg", bbox_inches="tight")

    return fig, inner_axes

In [7]:
dataset_prefix = "runtime/data"
dataset_postfix = "-ry-seed42-train1024-validation1024-test1024.npz"

task_names = [
    "fdgo",
    "fdanti",
    "reactgo",
    "reactanti",
    "delaygo",
    "delayanti",
    "dm1",
    "delaydm1",
    "dm2",
    "delaydm2",
    "contextdm1",
    "contextdelaydm1",
    "contextdm2",
    "contextdelaydm2",
    "multidm",
    "multidelaydm",
]

root_path = Path("..").expanduser().resolve()
out_path = Path("./output")
out_path.mkdir(parents=True, exist_ok=True)

ncols = 4
nrows = math.ceil(len(task_names) / ncols)

fig, outer_axes = plt.subplots(
    nrows,
    ncols,
    figsize=(ncols * 4, nrows * 4),
)

outer_axes = np.asarray(outer_axes).reshape(nrows, ncols)

datasets = {}
inner_axes_all = []

for i, task_name in enumerate(task_names):
    row = i // ncols
    col = i % ncols

    parent_ax = outer_axes[row, col]

    dataset = DataSetManager.load(
        f"{task_name}-ry", 42, 1024, 1024, 1024, root_path / dataset_prefix
    )
    datasets[dataset.key] = dataset

    X, Y, M = dataset.train_x, dataset.train_y, dataset.train_mask

    is_left_col = col == 0
    is_bottom_row = row == nrows - 1

    if task_name == "fdgo":
        # 1) 保存单独 PNG
        fig_single, inner_axes_single = save_trial_axes_png(
            X=X,
            Y=Y,
            M=M,
            out_path=out_path,
            title=task_name,
            trial_index=0,
            stimulus_1_start=1,
            stimulus_2_start=32,
            stimulus_2_end=64,
        )
        plt.close(fig_single)  # 很重要，避免开太多 figure

    # 2) 在总图对应 cell 里画内容
    inner_axes = plot_trial_axes(
        parent_ax=parent_ax,
        trial_index=0,
        X=X,
        Y=Y,
        M=M,
        stimulus_1_start=1,
        stimulus_2_start=32,
        stimulus_2_end=64,
        title=task_name,
        show_ylabels=is_left_col,
        show_xlabel=is_bottom_row,
    )

    inner_axes_all.append(inner_axes)
    print(dataset.key)

# 关掉多余的空 cell
for j in range(len(task_names), nrows * ncols):
    row = j // ncols
    col = j % ncols
    outer_axes[row, col].axis("off")

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.30, 0.93),
    ncol=5,
    frameon=False,
)
fig.legend(
    handles=color_legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.70, 0.93),
    ncol=5,
    frameon=False,
    handlelength=1,
    handleheight=1,
)

fig.subplots_adjust(wspace=0.15, hspace=0.32)
fig.savefig(out_path / "SupplementaryFigure1.png", dpi=300, bbox_inches="tight")
# fig.savefig(out_path / "SupplementaryFigure1.svg", bbox_inches="tight")
plt.close(fig)

E0804 14:33:13.516029 2453824 cuda_executor.cc:1182] [0] Failed to allocate device memory of 29.64GiB (31824281600 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0804 14:33:13.516104 2453824 cuda_executor.cc:1182] [0] Failed to allocate device memory of 26.67GiB (28641853440 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0804 14:33:13.516157 2453824 cuda_executor.cc:1182] [0] Failed to allocate device memory of 24.01GiB (25777668096 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0804 14:33:13.516206 2453824 cuda_executor.cc:1182] [0] Failed to allocate device memory of 21.61GiB (23199899648 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0804 14:33:13.516254 2453824 cuda_executor.cc:1182] [0] Failed to allocate device memory of 19.45GiB (20879908864 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0804 14:33:13.516301 2453824 cuda_executor.cc:1182] [0] Failed to allocate

fdgo-ry
fdanti-ry
reactgo-ry
reactanti-ry
delaygo-ry
delayanti-ry
dm1-ry
delaydm1-ry
dm2-ry
delaydm2-ry
contextdm1-ry
contextdelaydm1-ry
contextdm2-ry
contextdelaydm2-ry
multidm-ry
multidelaydm-ry
